In [0]:
data = [
    (101, "John",  "2024-01", 5000),
    (101, "John",  "2024-02", 5500),
    (101, "John",  "2024-03", 6000),
    (101, "John",  "2024-04", 5800),

    (102, "Mary",  "2024-01", 4000),
    (102, "Mary",  "2024-02", 4200),
    (102, "Mary",  "2024-03", 4500),
    (102, "Mary",  "2024-04", 4700),

    (103, "David", "2024-01", 7000),
    (103, "David", "2024-02", 6800),
    (103, "David", "2024-03", 7200),
    (103, "David", "2024-04", 7500)
]

df_1 = spark.createDataFrame(
    data,
    ["employee_id", "employee_name", "month", "salary"]
)

df_1.show()

In [0]:


w = Window.partitionBy("employee_id").orderBy("month")

df = df.withColumn("Previous_salary", lag("salary", 1, 0).over(w)) \
       .withColumn("Future_salary", lead("salary", 1, 0).over(w)) \
       .withColumn("salary_diff", col("Future_salary") - col("Previous_salary"))\
       .withColumn("_rn", row_number().over(w)).filter(col("_rn")==1).drop("_rn")

display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, lead, row_number, sum
drop_dublicate=df_1.dropDuplicates(["employee_id"])
# # drop_dublicate.show()
# df_1.show()
w_rt=Window.partitionBy(col("employee_id"),col("employee_name")).orderBy(col("month")).rowsBetween(Window.unboundedPreceding,Window.currentRow)
running_total=df_1.withColumn("Runnning_total", sum(col("salary")).over(w_rt))
running_total.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# =========================
# Sample Data
# =========================

data = [
    (101, "John",  "2024-01", 5000),
    (101, "John",  "2024-02", 5500),
    (101, "John",  "2024-03", 6000),
    (101, "John",  "2024-04", 5800),

    (102, "Mary",  "2024-01", 4000),
    (102, "Mary",  "2024-02", 4200),
    (102, "Mary",  "2024-03", 4500),
    (102, "Mary",  "2024-04", 4700),

    (103, "David", "2024-01", 7000),
    (103, "David", "2024-02", 6800),
    (103, "David", "2024-03", 7200),
    (103, "David", "2024-04", 7500)
]

df = spark.createDataFrame(
    data,
    ["employee_id", "employee_name", "month", "salary"]
)

# Base Window
w = Window.partitionBy("employee_id").orderBy("month")

# ==========================================================
# 1. LAG - Previous Salary
# ==========================================================

df.withColumn(
    "prev_salary",
    lag("salary").over(w)
).show()

# ==========================================================
# 2. LEAD - Next Salary
# ==========================================================

df.withColumn(
    "next_salary",
    lead("salary").over(w)
).show()

# ==========================================================
# 3. Month-over-Month Growth
# ==========================================================

df.withColumn(
    "prev_salary",
    lag("salary").over(w)
).withColumn(
    "growth_pct",
    round(
        ((col("salary") - col("prev_salary"))
         / col("prev_salary")) * 100,
        2
    )
).show()

# ==========================================================
# 4. Latest Record Per Employee
# ==========================================================

w_latest = Window.partitionBy("employee_id") \
                 .orderBy(col("month").desc())

df.withColumn(
    "rn",
    row_number().over(w_latest)
).filter(
    col("rn") == 1
).drop("rn").show()

# ==========================================================
# 5. First Record Per Employee
# ==========================================================

df.withColumn(
    "rn",
    row_number().over(w)
).filter(
    col("rn") == 1
).drop("rn").show()

# ==========================================================
# 6. Top 2 Salaries Per Employee
# ==========================================================

w_salary = Window.partitionBy("employee_id") \
                 .orderBy(col("salary").desc())

df.withColumn(
    "rn",
    row_number().over(w_salary)
).filter(
    col("rn") <= 2
).show()

# ==========================================================
# 7. Rank
# ==========================================================

df.withColumn(
    "rank",
    rank().over(w_salary)
).show()

# ==========================================================
# 8. Dense Rank
# ==========================================================

df.withColumn(
    "dense_rank",
    dense_rank().over(w_salary)
).show()

# ==========================================================
# 9. Running Total
# ==========================================================

w_running = Window.partitionBy("employee_id") \
                  .orderBy("month") \
                  .rowsBetween(
                      Window.unboundedPreceding,
                      Window.currentRow
                  )

df.withColumn(
    "running_total",
    sum("salary").over(w_running)
).show()

# ==========================================================
# 10. Running Maximum
# ==========================================================

df.withColumn(
    "running_max",
    max("salary").over(w_running)
).show()

# ==========================================================
# 11. Moving Average (Last 3 Rows)
# ==========================================================

w_moving = Window.partitionBy("employee_id") \
                 .orderBy("month") \
                 .rowsBetween(-2, 0)

df.withColumn(
    "moving_avg",
    round(avg("salary").over(w_moving), 2)
).show()

# ==========================================================
# 12. First Salary
# ==========================================================

df.withColumn(
    "first_salary",
    first("salary").over(w)
).show()

# ==========================================================
# 13. Last Salary
# ==========================================================

w_last = Window.partitionBy("employee_id") \
               .orderBy("month") \
               .rowsBetween(
                   Window.unboundedPreceding,
                   Window.unboundedFollowing
               )

df.withColumn(
    "last_salary",
    last("salary").over(w_last)
).show()

# ==========================================================
# 14. Difference From First Salary
# ==========================================================

df.withColumn(
    "first_salary",
    first("salary").over(w)
).withColumn(
    "salary_diff",
    col("salary") - col("first_salary")
).show()

# ==========================================================
# 15. Second Highest Salary Per Employee
# ==========================================================

df.withColumn(
    "rn",
    row_number().over(w_salary)
).filter(
    col("rn") == 2
).show()

# ==========================================================
# 16. Detect Salary Change
# ==========================================================

df.withColumn(
    "prev_salary",
    lag("salary").over(w)
).withColumn(
    "salary_changed",
    when(
        col("salary") != col("prev_salary"),
        "YES"
    ).otherwise("NO")
).show()

# ==========================================================
# 17. Duplicate Detection
# ==========================================================

w_dup = Window.partitionBy(
    "employee_id",
    "month"
)

df.withColumn(
    "cnt",
    count("*").over(w_dup)
).filter(
    col("cnt") > 1
).show()

# ==========================================================
# 18. Nth Highest Salary (Example: 3rd Highest)
# ==========================================================

df.withColumn(
    "rn",
    row_number().over(w_salary)
).filter(
    col("rn") == 3
).show()